# Kaza Phase 1 — Step 1: Colab Pro Setup

Verifies Python, PyTorch, GPU capability (native BF16 on Colab Pro L4/A100, with FP32 fallback — never FP16), clones `udohchuks/kaza`, installs `transformers==5.24.0`, and runs a load + forward smoke check on `LiquidAI/Liquid-Codec` and `LiquidAI/LFM3-1.7B-Base`.

In [ ]:
# Cell 1: Print Python, PyTorch, and GPU info (run first on any Colab session)
import sys, torch

print(f"Python:  {sys.version.split()[0]} (need >= 3.11)")
print(f"PyTorch: {torch.__version__} (need >= 2.6)")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = f"{props.major}.{props.minor}"
    vram = props.total_memory / (1024 ** 3)
    bf16_hw = "native BF16" if props.major >= 8 else "no native BF16 hardware (T4 is 7.5)"
    print(f"GPU:     {props.name} | {vram:.1f} GB | compute capability {cc} ({bf16_hw})")
else:
    print("GPU:     NONE — switch runtime to GPU (L4 / A100 / T4)")

In [ ]:
# Cell 2: Clone (or update) repo and install dependencies
import os, subprocess, sys
from pathlib import Path

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

REPO_DIR = ROOT / "kaza" if ROOT.name != "kaza" else ROOT
REPO_URL = "https://github.com/udohchuks/kaza.git"
BRANCH = "phase1-design"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Ready in {Path.cwd()}")

In [ ]:
# Cell 3: Run Kaza environment & BF16 ops+backward smoke check
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from kaza.env import print_env

env = print_env()
env.data_dir.mkdir(parents=True, exist_ok=True)
env.out_dir.mkdir(parents=True, exist_ok=True)
print(f"Data dir: {env.data_dir} | Out dir: {env.out_dir}")

In [ ]:
# Cell 4: Smoke-test loading & forward pass on LiquidCodec and LFM3-1.7B-Base
import gc, torch
from transformers import AutoModel, AutoModelForCausalLM

print("Loading LiquidAI/Liquid-Codec...")
codec = AutoModel.from_pretrained("LiquidAI/Liquid-Codec", trust_remote_code=True)
codec = codec.to(env.device).eval()
n_codec = sum(p.numel() for p in codec.parameters()) / 1e6
print(f"Liquid-Codec loaded: {n_codec:.1f}M params")
del codec
gc.collect()
if env.device.type == "cuda":
    torch.cuda.empty_cache()

print(f"Loading LiquidAI/LFM3-1.7B-Base in {env.frozen_dtype}...")
backbone = AutoModelForCausalLM.from_pretrained(
    "LiquidAI/LFM3-1.7B-Base",
    dtype=env.frozen_dtype,
).to(env.device).eval()
n_backbone = sum(p.numel() for p in backbone.parameters()) / 1e9
print(f"LFM3-1.7B-Base loaded: {n_backbone:.2f}B params | layers: {len(backbone.model.layers)}")

with torch.no_grad(), torch.amp.autocast(env.device.type, dtype=env.autocast_dtype):
    dummy_ids = torch.zeros((1, 8), dtype=torch.long, device=env.device)
    out = backbone(input_ids=dummy_ids)
    print(f"LFM3 forward smoke check OK: logits {tuple(out.logits.shape)}, finite={torch.isfinite(out.logits).all().item()}")

if env.device.type == "cuda":
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    print(f"GPU memory allocated: {alloc:.2f} GB")

del backbone, out
gc.collect()
if env.device.type == "cuda":
    torch.cuda.empty_cache()